# Privacy d'export et minage de motifs — `opaque_id`, `sanitize_state`, `pattern_mining`

**Navigation** : [README](./README.md) | [<< 08f-synthèse](./Argumentation-08f-Synthese-Multidimensionnelle-Python.ipynb)

> **Tranche T3 du pilotage #23163** — la série `Argument_Analysis` devient
> **complète sur CoursIA**. Ce carnet expose trois organes portés **verbatim**
> depuis le dépôt EPITA (`2025-Epita-Intelligence-Symbolique`, commit
> `2e2a1c3c`) : le générateur d'identifiants opaques, le garde d'export de
> l'état d'analyse, et le minage de motifs au-dessus des corpus.

**Durée estimée** : 35 minutes
**Prérequis** : aucun — Python standard uniquement, **aucun appel réseau, aucune clé, aucun JVM**.

## Objectifs

- Comprendre ce que garantit (et ce que ne garantit pas) un identifiant
  « opaque » : il ne se renverse pas, mais un **sel public** en fait un oracle
  de confirmation.
- Comprendre le contrat du garde d'export : **les agrégats survivent, les
  textes nominatifs partent**, et un conteneur **non connu traverse intact** —
  l'allowlist est le contrat, pas un filet de sécurité.
- Comprendre ce que le minage de motifs mesure **une fois l'état assaini** :
  des frequencies de sophismes par corpus, une asymétrie tricherie/influence,
  des co-occurrences, une couverture croisée, et trois détecteurs formels.

## Discipline du carnet

Toutes les données sont **synthétiques** : aucun corpus réel, aucun nom réel,
aucun identifiant réel, et le sel utilisé est **public et affiché** — c'est
délibéré (voir la section 1). Le carnet n'exerce que des chemins déterministes.

In [1]:
# Cellule [1] - Mise en place : modules portes, fixture, sel synthetique
#
# Le sel est SYNTHETIQUE et PUBLIC : il sert a reproduire les valeurs de la
# fixture amont. C'est le point pedagogique de la section 1, pas une fuite.
import copy
import json
import os
import sys
from pathlib import Path

# Rendre argumentation_lib importable quel que soit le cwd d'execution :
# on accepte aussi bien le dossier de la serie que la racine du depot.
candidate = Path.cwd()
aa_dir = None
for _ in range(6):
    for probe_dir in (candidate,
                      candidate / "MyIA.AI.Notebooks" / "SymbolicAI" / "Argument_Analysis"):
        if (probe_dir / "argumentation_lib" / "__init__.py").exists():
            aa_dir = probe_dir
            break
    if aa_dir is not None:
        break
    if len(candidate.parts) <= 1:
        break
    candidate = candidate.parent
if aa_dir is None:
    raise RuntimeError(
        "Dossier argumentation_lib/ introuvable depuis %r. Executez le carnet "
        "depuis MyIA.AI.Notebooks/SymbolicAI/Argument_Analysis/." % Path.cwd())
sys.path.insert(0, str(aa_dir))

# La fixture portee est cherchee par remontee : elle vit dans data/ du dossier
# de la serie, quel que soit le cwd reel du noyau.
fixture_path = None
probe = aa_dir
for _ in range(4):
    candidat = probe / "data" / "privacy_mining_examples.json"
    if candidat.exists():
        fixture_path = candidat
        break
    if probe.parent == probe:
        break
    probe = probe.parent
if fixture_path is None:
    raise FileNotFoundError(
        "Fixture privacy_mining_examples.json introuvable depuis %r" % (aa_dir,))

from argumentation_lib._opaque_id import opaque_id
from argumentation_lib._sanitize_state import sanitize_state
import argumentation_lib._pattern_mining as pm

EXAMPLES = json.loads(fixture_path.read_text(encoding="utf-8"))
assert EXAMPLES["asset"] == "privacy_mining"

# Sel synthetique, public, affiche : il reproduit les identifiants attendus
# par la fixture amont (memes valeurs que le carnet EPITA).
os.environ["OPAQUE_ID_SALT"] = "synthetic-asset-salt"

print("modules portes :", opaque_id.__module__, sanitize_state.__module__, pm.__name__)
print("fixture        :", fixture_path.name, "|", len(EXAMPLES["sanitize_cases"]), "cas de sanitisation,",
      len(EXAMPLES["mining_cases"]), "cas de minage")
print("sections       :", [k for k in EXAMPLES if k.endswith("_cases")])

modules portes : argumentation_lib._opaque_id argumentation_lib._sanitize_state argumentation_lib._pattern_mining
fixture        : privacy_mining_examples.json | 8 cas de sanitisation, 10 cas de minage
sections       : ['opaque_cases', 'sanitize_cases', 'mining_cases']


### Interprétation : ce que la mise en place établit

Trois points, tous vérifiables dans la sortie ci-dessus :

- Les trois organes sont **importables tels quels** : le portage n'a pas
  réécrit la sémantique, il a seulement changé le préfixe d'import d'un seul
  module (`_sanitize_state` → `_opaque_id`).
- La fixture **porte les valeurs attendues** de l'amont : c'est elle qui sert
  de témoin, pas le souvenir du carnet EPITA.
- Le sel est **synthétique et visible dans le code**. Un secret n'aurait pas sa
  place dans un carnet committé — et la section 1 montre précisément pourquoi
  un sel在这里 n'est pas un secret.

## 1. `opaque_id` — un identifiant qui ne se renverse pas, mais qui se confirme

`opaque_id(nom, sel)` rend les **8 premiers hex de `sha256(sel + nom)`**. Trois
propriétés, et une quatrième qui est la contrepartie :

| Propriété | Ce qu'elle signifie |
|---|---|
| déterministe | même (nom, sel) → même identifiant |
| distinct | deux noms distincts sous un même sel → identifiants distincts |
| non réversible | `sha256` n'a pas d'attaque par pré-image : on ne remonte pas au nom |
| **confirmable** | avec un sel **public**, qui devine un nom recalcule son identifiant en O(1) et **confirme** le match |

La quatrième propriété annule les trois autres quant à la confidentialité :
l'opacité n'est pas de l'obfuscation, c'est un **oracle de confirmation**.
C'est pourquoi le module **refuse de tourner sans sel explicite** (issue amont
#1973) : l'ancien sel par défaut vivait en clair dans le fichier — donc dans
l'historique du dépôt — et ne protégeait rien.

In [2]:
# Cellule [2] - Les trois proprietes, puis le refus sans sel
stored_opaque = {c["name"]: c for c in EXAMPLES["opaque_cases"]}
cas = stored_opaque["deterministic_and_distinct"]

a1 = opaque_id("Source A", salt="s1")
a2 = opaque_id("Source A", salt="s1")
b1 = opaque_id("Source B", salt="s1")
a3 = opaque_id("Source A", salt="s2")

print("Source A / s1 :", a1)
print("Source A / s1 :", a2, "<- stable")
print("Source B / s1 :", b1, "<- distinct")
print("Source A / s2 :", a3, "<- autre sel, tout change")

assert a1 == a2 == cas["id_a_1"] == cas["id_a_2"]
assert b1 == cas["id_b_same_salt"] != a1
assert a3 == cas["id_a_other_salt"] != a1

# Le refus : sans sel (ni argument, ni variable d'environnement), l'appel leve.
saved = os.environ.pop("OPAQUE_ID_SALT")
try:
    opaque_id("Source A")
    sans_sel = "AUCUNE EXCEPTION (BUG)"
except RuntimeError as exc:
    sans_sel = "RuntimeError : %s" % exc
finally:
    os.environ["OPAQUE_ID_SALT"] = saved
print("sans sel ->", sans_sel)
assert sans_sel.startswith("RuntimeError")

# Le sel de l'environnement est utilise quand l'argument est absent.
print("sel d'environnement ->", opaque_id("Source A"))
assert opaque_id("Source A") == opaque_id("Source A", salt="synthetic-asset-salt")

Source A / s1 : a295797e
Source A / s1 : a295797e <- stable
Source B / s1 : 5f602022 <- distinct
Source A / s2 : 193c2319 <- autre sel, tout change
sans sel -> RuntimeError : OPAQUE_ID_SALT is required: no salt argument was passed and the OPAQUE_ID_SALT environment variable is unset. A public default would let any reader of the repo confirm a guessed name against a published opaque ID; set OPAQUE_ID_SALT in your .env / CI secrets instead.
sel d'environnement -> 897cdd36


### Interprétation : deux choses que la mesure dit mieux que la prose

1. **La stabilité est totale, la variation est totale.** Changer le sel change
   *tous* les identifiants : un identifiant opaque n'est pas une étiquette
   stable dans le temps, il est stable **pour un couple (nom, sel)**. C'est
   pourquoi il ne peut pas servir de clé de jointure entre deux exécutions
   exécutées avec des sels différents.
2. **Le refus est une garantie, pas une commodité.** `RuntimeError` sans sel
   signifie qu'un appel ne peut pas produire d'identifiant « par défaut » —
   donc qu'un identifiant publié ne peut pas être réémis avec un sel
   différent sans casser les références existantes.

La contrepartie est à écrire dans le cours : avec un sel **public** (celui de
ce carnet), l'identifiant de « Source A » est reproductible par n'importe qui.
L'opacité protège contre la *lecture directe* du nom, pas contre un adversaire
qui connaît le sel.

## 2. `sanitize_state` — l'allowlist qui laisse passer les agrégats

Le garde d'export prend l'état complet d'une analyse et en rend une version
**dépouillée**. Son contrat tient en une phrase :

> **Tout contenu quantitatif ou topologique survit, tout texte nominatif
> part.**

Les passes sont numérotées dans le code amont ; on les suit ici par leur
effet observable :

| Passe | Effet |
|---|---|
| 1 | retrait des champs nominatifs de premier niveau (`source_name`, `author`, `raw_text`, `url`…) |
| 2 / 2b | opacification de l'identifiant de source, et des **valeurs** de métadonnées (les **clés** survivent) |
| 3 / 4 | un argument **texte** devient `{"text_stripped": true}` ; un argument **structuré** passe intact ; un champ narratif (`justification`, `llm_assessment`) part, son voisin numérique reste |
| 4b-4d | la **topologie Dung** survit (arêtes, nombre et taille des extensions) ; seuls les **membres** sont opacifiés |
| 5b-5f | les conteneurs `List[Dict]` : mêmes règles, un niveau plus bas |
| 6 | les symboles sont remplacés |
| 7 | le narratif devient `{length, stripped}` |
| 8 | le struct devient des **compteurs** et un drapeau de présence |
| 9 | la **projection de surplus** (`zero_shot_surplus`) : une forme non conforme est remplacée par un marqueur `{"scrubbed": ...}` |

Deux propriétés de conception dominent tout le reste :

- **C'est une allowlist.** Un conteneur inconnu **traverse intact**. Ce n'est
  pas un oubli : un garde fondé sur une liste de clés à filtrer laisserait
  passer tout ce qu'on n'a pas prévu ; une allowlist l'envoie dans le vide.
  Le prix est explicite : le carnet le montre, et il est assumé.
- **L'entrée n'est pas mutée.** Le garde travaille sur une copie profonde.

In [3]:
# Cellule [3] - Un etat d'analyse synthetique, et son passage au garde
stored_san = {c["name"]: c for c in EXAMPLES["sanitize_cases"]}

STATE = {
    "source_name": "Discours de synthèse (nom réel)",
    "author": "Auteur Réel",
    "url": "https://example.invalid/doc",
    "raw_text": "Texte brut long du document source.",
    "source_id": "doc_source_001",
    "source_metadata": {"genre": "discours", "title": "Titre réel du discours", "channel": "TV"},
    "identified_arguments": {"arg1": "Le premier argument en clair", "arg2": {"score": 3}},
    "identified_fallacies": {"f1": {"type": "ad_hominem", "justification": "cite le texte réel", "severity": 2}},
    "argument_quality_scores": {"arg1": {"overall": 0.8, "llm_assessment": "narratif LLM citant l'argument"}},
    "dung_frameworks": {"fw1": {
        "name": "dung_grounded",
        "arguments": ["le premier argument en clair", "le second en clair"],
        "attacks": [["le premier argument en clair", "le second en clair"]],
        "extensions": {"extensions": [["le premier argument en clair"]], "count": 1, "sizes": [1],
                       "all_members": ["le premier argument en clair"]},
        "formalism_specific": {
            "contraries": {"atome_a": "contraire_a"},
            "attack_weights": [{"source": "atome_a", "target": "atome_b", "weight": 0.7}],
            "criterion": "generalized_specificity",
        },
    }},
    "counter_arguments": [{"strategy": "reductio", "counter_content": "contre-argument en clair", "score": 0.5}],
    "extracts": [{"content": "extrait en clair", "claim_id": "c1"}],
    "probabilistic_results": [{"method": "ml", "arguments": ["argument en clair"],
                               "acceptance_probabilities": {"argument en clair": 0.42}}],
    "dialogue_results": [{"topic": "sujet en clair",
                           "trace": [{"round": 1, "speaker": "S1", "argument": "texte en clair",
                                      "target": "autre texte"}]}],
    "debate_transcripts": [{"round": 1, "topic": "sujet débat",
                            "exchanges": [{"point": "point en clair", "rebuttal": "réponse en clair",
                                           "scheme": "expert_opinion"}]}],
    "nl_to_logic_translations": [{"source": "s", "logic": "P->Q", "original_text": "texte original",
                                   "variables": {"p": "signification en clair de p"}}],
    "belief_revision_results": [{"method": "m", "original": ["croyance en clair"], "revised": [],
                                 "minimal_retraction": {"options": [["croyance en clair"]], "cardinality": 1}}],
    "narrative_synthesis": "Long texte narratif de synthèse.",
    "final_conclusion": "Conclusion finale narrative.",
    "stakes_and_stakeholders": {"stakes": ["enjeu en clair"], "stakeholders": ["acteur en clair"],
                                "rhetorical_register": "délibératif"},
    "identified_metrics": {"n_args": 2, "score": 0.9},
    "unknown_new_container": [{"free": "texte d'un conteneur non classé"}],
}
original = copy.deepcopy(STATE)
san = sanitize_state(STATE)

print("cles de l'etat :", len(STATE), "->", len(san))
for champ in ("source_name", "author", "raw_text"):
    print("  %-12s retire : %s" % (champ, champ not in san))

s1 = stored_san["top_level_stripped_and_source_opacified"]
assert "source_name" not in san and not s1["source_name_present"]
assert "author" not in san and not s1["author_present"]
assert "raw_text" not in san and not s1["raw_text_present"]
print("source_id      :", san["source_id"])
assert san["source_id"] == s1["source_id_after"] and s1["source_id_is_hex"]
print("source_metadata:", san["source_metadata"])
assert list(san["source_metadata"].keys()) == s1["metadata_keys_after"]
assert list(san["source_metadata"].values()) == s1["metadata_values_after"]

cles de l'etat : 22 -> 18
  source_name  retire : True
  author       retire : True
  raw_text     retire : True
source_id      : 3599baaa
source_metadata: {'genre': '5b656822', 'title': '8a9e44dd', 'channel': '6fc47b79'}


### Interprétation : ce qui disparaît, ce qui reste — et ce que ça prouve

- Les **clés** de `source_metadata` survivent (`genre`, `title`, `channel`) :
  on garde **quelles** métadonnées existaient. Les **valeurs** sont opacifiées.
  C'est le bon régime — la structure est l'information agrégée, la valeur est
  l'identification.
- `source_id` devient 8 hex : l'identifiant de source reste **référençable**
  sans être lisible.
- `identified_metrics` — un compteur et un score — traverse **intact**, et
  c'est mesuré plus bas (`identified_metrics` inchangé).

La conséquence pratique : un état assaini reste exploitable par les analyses
agrégées, mais ne peut pas être relu pour nommer qui a parlé.

### 2.2 — Passes 3 et 4 : le texte part, la structure reste

Un argument **chaîne** devient `{"text_stripped": true}` — on garde le fait
qu'il y avait un argument, pas ce qu'il disait. Un argument **structuré** (un
dictionnaire de mesures) traverse intact. Dans un sophisme, `type` (vocabulaire
fermé) et `severity` (nombre) survivent ; `justification` (un narratif qui cite
le texte) part. Même sort pour `llm_assessment` — et son voisin `overall`,
quantitatif, reste.

In [4]:
# Cellule [4] - Texte depouille, structure et mesures conservees
s2 = stored_san["arguments_and_justifications"]

assert san["identified_arguments"]["arg1"] == s2["string_argument_becomes"] == {"text_stripped": True}
assert san["identified_arguments"]["arg2"] == s2["structured_argument_survives"] == {"score": 3}
assert san["identified_fallacies"]["f1"] == s2["fallacy_after"]
assert "justification" not in san["identified_fallacies"]["f1"]
assert "llm_assessment" not in san["argument_quality_scores"]["arg1"]
assert san["argument_quality_scores"]["arg1"]["overall"] == 0.8

print("arg1 (texte)   ->", san["identified_arguments"]["arg1"])
print("arg2 (structure)->", san["identified_arguments"]["arg2"])
print("sophisme f1    ->", san["identified_fallacies"]["f1"])
print("qualite arg1   ->", san["argument_quality_scores"]["arg1"])

# Le texte nominatif a disparu de TOUT l'etat assaini, pas seulement des champs connus.
blob = json.dumps(san, ensure_ascii=False)
for motif in ("Le premier argument en clair", "Auteur Réel", "Titre réel du discours",
              "texte original", "signification en clair de p"):
    assert motif not in blob, motif
print("-> aucun des", 5, "motifs nominatifs du corpus de test ne subsiste dans l'etat assaini")

arg1 (texte)   -> {'text_stripped': True}
arg2 (structure)-> {'score': 3}
sophisme f1    -> {'type': 'ad_hominem', 'severity': 2}
qualite arg1   -> {'overall': 0.8}
-> aucun des 5 motifs nominatifs du corpus de test ne subsiste dans l'etat assaini


### Interprétation : la règle, et sa limite lisible

La règle est nette : **un champ textuel devient un marqueur de présence, un
champ numérique reste un nombre**. `{"text_stripped": true}` n'est pas une
valeur — c'est la **constatation** qu'il y avait un texte.

La limite se voit dans le test final : on vérifie que cinq motifs nominatifs
n'apparaissent **nulle part** dans l'état assaini, pas seulement dans les
champs que le carnet a énumérés. C'est la seule formulation qui ait du sens :
le garde est une allowlist, donc *ce qu'il ne connaît pas* traverse — et la
seule protection restante est **l'absence de contenu nominatif dans les
conteneurs connus**.

### 2.3 — Passes 4b à 4d : la topologie Dung survit à l'opacification

C'est le point le plus intéressant du garde : les **membres** d'un cadre
formel sont du texte (opacifié), mais la **topologie** est une donnée
quantitative (conservée). L'arité des attaques, le nombre et la taille des
extensions, le critère du formalisme et le poids d'une attaque : tout survit.
Sans cela, on ne pourrait plus raisonner sur l'état exporté — et c'est
justement ce qu'on veut pouvoir faire.

In [5]:
# Cellule [5] - La topologie survit, les membres sont opacifies
s3 = stored_san["topology_survives_opacification"]
fw = san["dung_frameworks"]["fw1"]

assert len(fw["attacks"][0]) == s3["attacks_arity"] == 2
assert fw["extensions"]["count"] == s3["extensions_count_kept"] == 1
assert fw["extensions"]["sizes"] == s3["extensions_sizes_kept"] == [1]
assert fw["name"] == s3["name_kept"] == "dung_grounded"
assert fw["formalism_specific"]["criterion"] == s3["criterion_kept"]
assert fw["formalism_specific"]["attack_weights"][0]["weight"] == s3["weight_kept"] == 0.7

print("attacks    :", fw["attacks"])
print("extensions :", fw["extensions"])
print("contraries :", fw["formalism_specific"]["contraries"])
assert "le premier argument en clair" not in json.dumps(san, ensure_ascii=False)
print("-> arite, compte, tailles, critere et poids conserves ; les membres, opacifies")

attacks    : [['d4073163', '0bbe670b']]
extensions : {'extensions': [['d4073163']], 'count': 1, 'sizes': [1], 'all_members': ['d4073163']}
contraries : {'f72b5fab': '0021a491'}
-> arite, compte, tailles, critere et poids conserves ; les membres, opacifies


### Interprétation : pourquoi l'ordre des passes n'est pas indifférent

L'attaque est conservée **par position**, pas par contenu : `attacks[0]` est un
couple de deux éléments, et c'est ce couple — donc l'**arête** — qui est
préservé. Ce qui change, c'est l'**étiquette** de ses extrémités.

Autrement dit : la relation de conflit survit, l'identité des arguments ne
survit pas. C'est exactement ce qu'un agrégat statisticien peut exploiter sans
rien savoir des personnes concernées, et c'est aussi la limite : deux analyses
distinctes ne sont plus joignables par leurs arguments, seulement par leur
structure.

### 2.4 — Passes 5b à 5f : les conteneurs `List[Dict]`

Même logique, un niveau plus bas. Dans une liste d'objets, certains champs
sont **fermé** (`method`, `scheme`, `speaker`, `strategy`) et survivent ; les
autres sont opacifiés. Cas le plus instructif : `acceptance_probabilities` est
une **table dont les clés sont du texte** (l'argument) et les **valeurs sont
des nombres** (la probabilité) : les clés sont opacifiées, les valeurs gardées
— la distribution survit, sa correspondance avec les arguments non.

In [6]:
# Cellule [6] - Un niveau plus bas : cles fermees, valeurs numeriques, cles opacifiees
s4 = stored_san["list_containers"]

assert san["counter_arguments"][0] == s4["counter_arguments_after"]
assert san["extracts"][0] == s4["extracts_after"]
assert san["probabilistic_results"][0]["method"] == s4["prob_method_kept"]
assert san["probabilistic_results"][0]["arguments"] == s4["prob_arguments_opaque"]
assert list(san["probabilistic_results"][0]["acceptance_probabilities"].keys()) == s4["prob_mapping_keys_opaque"]
assert list(san["probabilistic_results"][0]["acceptance_probabilities"].values()) == s4["prob_values_kept"] == [0.42]
assert san["dialogue_results"][0]["trace"][0]["speaker"] == s4["dialogue_speaker_kept"] == "S1"

print("counter_arguments :", san["counter_arguments"][0])
print("extracts          :", san["extracts"][0])
print("probabilites      :", san["probabilistic_results"][0])
print("trace (speaker)   :", san["dialogue_results"][0]["trace"][0])

counter_arguments : {'strategy': 'reductio', 'score': 0.5}
extracts          : {'claim_id': 'c1'}
probabilites      : {'method': 'ml', 'arguments': ['fb22732b'], 'acceptance_probabilities': {'fb22732b': 0.42}}
trace (speaker)   : {'round': 1, 'speaker': 'S1', 'argument': '2ba6db84', 'target': '9ef093ad'}


### Interprétation : la table qui survit sans ses clés

`acceptance_probabilities` est l'exemple le plus parlant de tout le carnet :
c'est une **table de correspondance** entre un argument (texte) et une
probabilité (nombre). Après passage, on a encore une table
`{opaque: 0.42}` — la **distribution** est intacte, la **correspondance** ne
l'est plus. Un usage statisticien (moyenne, écart-type, classement) reste
possible ; un usage de jointure ne l'est plus.

C'est le compromis central du garde : on **agrège au lieu de relire**.

### 2.5 — Passes 7 et 8, l'allowlist, et la copie profonde

Trois faits se mesurent ici :

- le **narratif** devient `{length, stripped}` — sa longueur suffit pour un
  indicateur de volume ;
- le **struct** devient des **compteurs** et un drapeau de présence
  (`has_rhetorical_register`) ;
- un conteneur **inconnu traverse intact** — `unknown_new_container` revient
  tel quel. L'allowlist ne devine pas : elle ne s'occupe que de ce qu'elle
  connaît, et **renvoie le reste à l'appelant** ;
- l'**entrée n'est pas mutée** : le garde travaille sur une copie profonde.

In [7]:
# Cellule [7] - Narratif, struct, conteneur inconnu, non-mutation
s5 = stored_san["narrative_and_struct_reductions"]
s6 = stored_san["unknown_container_traverses_intact"]
s7 = stored_san["input_not_mutated"]
s8 = stored_san["salt_requirement_is_conditional"]

assert san["narrative_synthesis"] == s5["narrative_after"]
assert san["narrative_synthesis"]["length"] == s5["narrative_length"] == len("Long texte narratif de synthèse.")
assert san["stakes_and_stakeholders"] == s5["stakes_after"]
assert san["stakes_and_stakeholders"]["has_rhetorical_register"] is True

assert san["unknown_new_container"] == s6["unknown_container_after"]
assert san["identified_metrics"] == s6["metrics_untouched"] == {"n_args": 2, "score": 0.9}
print("conteneur inconnu apres :", san["unknown_new_container"], "<- INTACT, par design")

assert STATE == original and s7["original_intact"] is True
print("l'entree n'est PAS mutee (copie profonde)")

# Le sel n'est exige que si un identifiant doit etre produit.
etat_sans_opacifiable = s8["state_without_opacifiable_fields"]
saved = os.environ.pop("OPAQUE_ID_SALT")
try:
    sans_opacifiable = sanitize_state(etat_sans_opacifiable)
finally:
    os.environ["OPAQUE_ID_SALT"] = saved
print("sans sel, etat sans champ opacifiable :", sans_opacifiable, "<- passe")
assert sans_opacifiable == etat_sans_opacifiable

saved = os.environ.pop("OPAQUE_ID_SALT")
try:
    sanitize_state({"source_id": "doc_source_001"})
    erreur = "AUCUNE EXCEPTION (BUG)"
except RuntimeError:
    erreur = "RuntimeError"
finally:
    os.environ["OPAQUE_ID_SALT"] = saved
print("sans sel, etat qui DOIT produire un identifiant :", erreur)
assert erreur == s8["state_with_source_id"]

conteneur inconnu apres :

 [{'free': "texte d'un conteneur non classé"}] <- INTACT, par design
l'entree n'est PAS mutee (copie profonde)
sans sel, etat sans champ opacifiable : {'identified_metrics': {'n': 1}} <- passe
sans sel, etat qui DOIT produire un identifiant : RuntimeError


### Interprétation : l'allowlist est un contrat, pas un filet

Le cas `unknown_new_container` est le plus important du carnet, et il est
**contre-intuitif** : un contenu named « texte d'un conteneur non classé »
traverse le garde **intact**. Un lecteur attendrait l'inverse.

La raison est une décision de sécurité, pas un oubli : un garde construit sur
une **liste de ce qu'il faut retirer** (`denylist`) laisse passer tout ce qu'on
n'a pas prévu — le champ inventé demain par un autre agent. Une **allowlist**
fait l'inverse : ce qu'elle ne connaît pas sort du périmètre de la garde, et
l'appelant doit le traiter. Le prix — un conteneur neuf traverse — est le
signal qui force l'appelant à le regarder.

Le garde n'est donc **pas** une promesse « aucune donnée nominative ne sort ».
C'est une promesse « **tout ce que je connais est traité** ». La sécurité
d'ensemble se gagne par l'**agrégation en amont** : on n'exporte que des
agrégats, pas un état.

## 3. `pattern_mining` — les motifs au-dessus des corpus

Le minage de motifs consomme des **signatures** `{metadata, state}` : les
états d'analyse, **déjà assainis**. Il produit :

- `fallacy_spectrum` : la fréquence des types de sophisme par corpus ;
- `trick_vs_influence_ratio` : une asymétrie entre les **tricheries**
  (8 types — l-auto-biais qui arrange l'émetteur) et les **influences**
  (12 types — le procédé rhétorique qui arrange le récepteur) ;
- `cooccurrence_matrix` : les paires de sophismes qui reviennent ensemble, avec
  support, confiance, lift et Jaccard ;
- `cross_coverage` : la couverture croisée entre l'informel et le formel ;
- `run_formal_detectors` : trois détecteurs formels (topologie Dung, branchement
  ATMS, retraitement JTMS).

In [8]:
# Cellule [8] - Le spectre de sophismes par corpus
stored_mining = {c["name"]: c for c in EXAMPLES["mining_cases"]}

SIGS = [
    {"metadata": {"cluster_id": "corpus_A"}, "state": {
        "identified_fallacies": {
            "f1": {"type": "ad_hominem", "source_arg": "arg1"},
            "f2": {"type": "straw_man", "source_arg": "arg1"},
            "f3": {"type": "cherry_picking", "source_arg": "arg2"},
        },
        "fol_analysis_results": [{"formula": "F", "valid": False}],
        "dung_frameworks": {"fw": {"attacks": [{"from": "a", "to": "b"}]}},
        "jtms_retraction_chain": [{"trigger": "x"}],
    }},
    {"metadata": {"cluster_id": "corpus_B"}, "state": {
        "identified_fallacies": {
            "f1": {"type": "appeal_to_authority", "source_arg": "arg1"},
            "f2": {"type": "false_cause", "source_arg": "arg2"},
        },
        "dung_frameworks": {"fw": {"attacks": [{"from": "a", "to": "b"}, {"from": "b", "to": "c"}]}},
    }},
]

spec = pm.fallacy_spectrum(SIGS)
print(json.dumps(spec, ensure_ascii=False, indent=1))
assert spec == stored_mining["spectrum"]["result"]

total = sum(spec["corpus_A"].values())
print("somme des frequences corpus_A :", total, "(frequences arrondies a 4 decimales)")
# Trois tiers arrondis a 4 decimales -> 0.9999, pas 1.0 exact : l'artefact est mesure.
assert abs(total - 1.0) < 2e-4

{
 "corpus_A": {
  "ad_hominem": 0.3333,
  "straw_man": 0.3333,
  "cherry_picking": 0.3333
 },
 "corpus_B": {
  "appeal_to_authority": 0.5,
  "false_cause": 0.5
 }
}
somme des frequences corpus_A : 0.9999 (frequences arrondies a 4 decimales)


### Interprétation : une fréquence de corpus, pas une vérité sur un texte

Le spectre donne, pour chaque corpus, la part de chaque type de sophisme. Deux
remarques que la mesure impose :

- Les fréquences sont **arrondies à 4 décimales**, donc leur somme vaut
  `0.9999` et non `1.0` exactement. Le test tolère `2e-4` : c'est un artefact
  d'arrondi connu, pas une approximation de façade.
- Le corpus_A a **trois sophismes**, le corpus_B **deux** : la fréquence est
  normalisée par corpus. Deux corpus de tailles différentes ne sont donc pas
  directement comparables sur ce tableau — il faut passer au support des
  co-occurrences (section 3.3) pour une lecture de volume.

In [9]:
# Cellule [9] - Asimetrie tricherie / influence, et son plafond
tvi = pm.trick_vs_influence_ratio(SIGS)
print(json.dumps(tvi, ensure_ascii=False, indent=1))
assert tvi == stored_mining["trick_vs_influence"]["result"]

# corpus_A : 2 influence (ad_hominem, straw_man) vs 1 tricherie (cherry_picking)
assert tvi["corpus_A"]["asymmetry"] == round((2 - 1) / (2 + 1), 4)

edge = stored_mining["trick_vs_influence_edges"]
only_inf = pm.trick_vs_influence_ratio([
    {"metadata": {"cluster_id": "only_influence"},
     "state": {"identified_fallacies": {"f": {"type": "bandwagon", "source_arg": "a"}}}}])
print("tricherie=0 :", only_inf["only_influence"])
assert only_inf == edge["only_influence"]
assert only_inf["only_influence"]["ratio"] == 1e9  # plafonne, pas de division par zero

{
 "corpus_A": {
  "tricherie_share": 0.3333,
  "influence_share": 0.6667,
  "ratio": 2.0,
  "asymmetry": 0.3333
 },
 "corpus_B": {
  "tricherie_share": 0.5,
  "influence_share": 0.5,
  "ratio": 1.0,
  "asymmetry": 0.0
 }
}
tricherie=0 : {'tricherie_share': 0.0, 'influence_share': 1.0, 'ratio': 1000000000.0, 'asymmetry': 1.0}


### Interprétation : deux tables qui ne disent pas la même chose

La table `TRICHERIE_TYPES` (8 types) et la table `INFLUENCE_TYPES` (12 types)
ne se recouvrent pas : elles classent le sophisme selon **qui** il sert — le
biais qui arrange l'émetteur (tricherie) ou le procédé qui arrange le
récepteur (influence).

L'asymétrie `(influence - tricherie) / (influence + tricherie)` est donc un
**indicateur de biais d'émetteur** : positive quand le corpus manipule surtout
le lecteur, négative quand il se sert surtout de lui. Sur corpus_A elle vaut
`1/3 ≈ 0.3333`.

Le cas limite est instructif : un corpus sans aucune tricherie ne produit pas
une division par zéro, mais un **plafond** de `1e9`. Un ratio plafonné n'est
pas un ratio — il faut le lire comme « asymétrie totale vers l'influence », et
jamais le moyener avec des ratios ordinaires.

In [10]:
# Cellule [10] - Co-occurrences : support, confiance, lift, Jaccard
co_arg = pm.cooccurrence_matrix(SIGS, unit="argument")
print(json.dumps(co_arg, ensure_ascii=False, indent=1))
assert co_arg == stored_mining["cooccurrence_argument_unit"]["result"]

# unite argument : arg1 de corpus_A porte ad_hominem + straw_man -> paire co-occurrente
assert any(p["a"] == "ad_hominem" and p["b"] == "straw_man" for p in co_arg["pairs"])
top = co_arg["pairs"][0]
print("top paire par lift :", top["a"], "+", top["b"], "lift =", top["lift"])

{
 "pairs": [
  {
   "a": "ad_hominem",
   "b": "straw_man",
   "support": 1,
   "confidence": 1.0,
   "lift": 4.0,
   "jaccard": 1.0
  }
 ],
 "unit_count": 4
}
top paire par lift : ad_hominem + straw_man lift = 4.0


### Interprétation : la co-occurrence se lit par son unité

L'option `unit` change ce que « une occurrence » veut dire : ici
`unit="argument"` — une co-occurrence est **un argument qui porte deux
sophismes**. Le même couple de sophismes peut alors apparaître sur des
arguments différents, et le paramètre change le support.

Les quatre indicateurs ne sont pas interchangeables :

- le **support** compte les co-occurrences ;
- la **confiance** est la proportion des cas où le couple apparaît parmi
  ceux où l'un des deux seulement apparaît ;
- le **lift** mesure l'écart à l'indépendance (supérieur à 1 = les deux vont
  ensemble plus que le hasard ne le prévoit) ;
- le **Jaccard** mesure le recouvrement des deux ensembles d'arguments.

C'est le **lift** qui répond à la question « ces deux sophismes sont-ils
liés ? », et non le support — qui ne répond qu'à « sont-ils fréquents ? ».

In [11]:
# Cellule [11] - Couverture croisee : l'informel croise avec le formel
cc = pm.cross_coverage(SIGS)
assert cc == stored_mining["cross_coverage"]["result"]
print("corpus_B appeal_to_authority :", json.dumps(cc["appeal_to_authority"], ensure_ascii=False))

# corpus_B : pas de FOL, pas de JTMS, mais c est attaque sans attaquer -> unsupported
assert cc["appeal_to_authority"]["per_signature_rate"]["dung_unsupported"] == 1.0
assert cc["appeal_to_authority"]["per_signature_rate"]["fol_invalid"] == 0.0
print("-> per_signature compte chaque signature une fois ; per_occurrence chaque occurrence")

corpus_B appeal_to_authority : {"per_signature_rate": {"fol_invalid": 0.0, "dung_unsupported": 1.0, "jtms_retraction": 0.0}, "per_occurrence_rate": {"fol_invalid": 0.0, "dung_unsupported": 1.0, "jtms_retraction": 0.0}}
-> per_signature compte chaque signature une fois ; per_occurrence chaque occurrence


### Interprétation : deux dénominateurs, deux questions

La couverture croisée pose une question simple — *ce sophisme informel est-il
soutenu ou contredit par l'analyse formelle ?* — et y répond par **deux
dénominateurs** :

- `per_signature` compte chaque **signature** une fois : « dans ce corpus, ce
  sophisme est-il contesté ? » ;
- `per_occurrence` compte chaque **occurrence** : « cette fois qu'il apparaît,
  est-il contesté ? ».

Les deux ne coïncident pas : un sophisme récurrent dans un petit nombre de
signatures produit des taux différents. Sur corpus_B, `dung_unsupported` vaut
`1.0` en `per_signature` : l'argument est attaqué sans lui-même attaquer — ce
que le cadre Dung qualifie d'**insoutenu**.

Et `fol_invalid` vaut `0.0` — non parce que la formule serait valide, mais
parce que **corpus_B ne porte aucune analyse FOL** : l'absence de mesure n'est
pas une mesure favorable. Confondre les deux, c'est lire « rien à signaler »
là où il n'y a rien eu à regarder.

In [12]:
# Cellule [12] - Les trois detecteurs formels, et leurs pieges mesures
det = pm.run_formal_detectors(SIGS[0])
assert det == stored_mining["formal_detectors"]["result"]
print("sur SIGS[0] (attaques en dicts) :", json.dumps(det["dung_topology"], ensure_ascii=False))
print("-> n_attacks compte des ENTREES, pas des aretes : piege mesure")

topo_sig = {"state": {"dung_frameworks": {"fw": {
    "arguments": ["a", "b", "c"],
    "attacks": [["a", "b"], ["b", "c"]],
    "extensions": {"grounded": ["a", "c"], "preferred": [["a", "c"]]},
}}}}
topo = pm.DungTopologyDetector().detect(topo_sig)
print("aretes en paires :", json.dumps(topo, ensure_ascii=False))
assert topo == stored_mining["dung_topology_directed_density"]["result"]
assert topo["density"] == stored_mining["dung_topology_directed_density"]["density_value"] == round(2 / 6, 4)
print("-> 2 aretes / (3*2) directed =", topo["density"], "(pas n(n-1)/2)")

# Le contrat tri-etats du branchement ATMS (#1650) : absent n'est pas "faux".
import logging
logging.disable(logging.WARNING)
atms = pm.AtmsBranchingDetector().detect({"state": {"atms_contexts": [
    {"assumptions": ["p", "q"], "coherent": True},
    {"assumptions": ["r"], "coherent": False},
    {"assumptions": []},  # pas de cle "coherent" -> non classe, PAS contradiction
]}})
logging.disable(logging.NOTSET)
print("atms 3 etats :", json.dumps(atms))
assert atms == stored_mining["atms_coherent_three_state_1650"]["result"]
assert atms["contradiction_rate"] == round(1 / 3, 4)  # 1/3 : le non-classe est exclu

# Un detecteur qui leve rend un drapeau, il ne fait pas tomber l'analyse.
class Boom:
    name = "boom"

    def detect(self, signature):
        raise ValueError("kaboom")


# Le journal du module porterait la traceback (chemins de la machine) : on le
# coupe pendant la demonstration -- le contrat indique reste dans la sortie.
logging.disable(logging.WARNING)
res = pm.run_formal_detectors({}, detectors=[Boom()])
logging.disable(logging.NOTSET)
assert res == stored_mining["detector_exception_is_error_flag"]["result"] == {"boom": {"error": 1.0}}
print("detecteur qui leve :", res, "<- drapeau, pas de crash")

sur SIGS[0] (attaques en dicts) : {"n_args": 0.0, "n_attacks": 1.0, "density": 1.0, "n_extensions": 0.0, "max_extension_size": 0.0}
-> n_attacks compte des ENTREES, pas des aretes : piege mesure
aretes en paires : {"n_args": 3.0, "n_attacks": 2.0, "density": 0.3333, "n_extensions": 2.0, "max_extension_size": 2.0}
-> 2 aretes / (3*2) directed = 0.3333 (pas n(n-1)/2)
atms 3 etats : {"max_assumption_count": 2.0, "avg_assumptions": 1.0, "contradiction_rate": 0.3333}
detecteur qui leve : {'boom': {'error': 1.0}} <- drapeau, pas de crash


### Interprétation : un détecteur qui se trompe doit le dire

Trois pièges mesurés, tous de la même famille — **un détecteur qui ne sait
pas** :

1. **Attaques en dictionnaires.** `attacks: [{"from": "a", "to": "b"}]` n'est pas
   la même forme que `attacks: [["a", "b"]]`. Dans la première forme,
   `n_attacks` compte des **entrées**, et la densité lit `1.0` — un artefact,
   pas une mesure. Dans la seconde forme (arêtes en paires), la densité vaut
   `2 / (3 × 2) = 0.3333` : elle est **dirigée**, pas `n(n-1)/2`.
2. **Le contrat tri-états du branchement ATMS** : `coherent=False` signifie
   « contradiction », `coherent` **absent** signifie « non classé » — le
   détecteur exclut et **signale**, il ne fabrique jamais une réponse par
   défaut à partir d'une absence.
3. **Un détecteur qui lève rend `{"error": 1.0}`**, jamais un crash : l'appelant
   voit l'échec dans les données, il nerompt pas l'analyse.

C'est la même exigence que le chemin d'échec du carnet 09 (garde-fous) : un
outil qui ne sait pas doit le dire, sinon l'appelant prend une absence de
mesure pour une mesure.

## Exercice 1 — Écrire une passe d'opacification

**Objectif** : écrire une fonction `opacifier_valeurs(etat, cles)` qui
parcourt un état et remplace **la valeur** de chaque clé de `cles` par son
identifiant opaque `opaque_id(...)`, en laissant les autres clés intactes.

**Contraintes** : ne modifie pas l'état d'entrée (travaille sur une copie
profonde) ; une clé absente de l'état ne doit pas créer d'erreur ; une clé
dont la valeur n'est pas une chaîne doit rester inchangée.

**Indice** : `copy.deepcopy` pour la copie, `isinstance(valeur, str)` pour le
garde-fou, et le sel synthétique de la cellule [1].

In [13]:
# Exercice 1 - a completer
def opacifier_valeurs(etat, cles):
    """Remplace la valeur de chaque cle presente dans `etat` par son identifiant opaque.

    TODO etudiant : faire le travail.
    """
    # TODO etudiant : implementer (copie profonde + garde-fou sur le type).
    return None  # TODO etudiant


print("Exercice 1 : a completer ->", opacifier_valeurs({"arg1": "texte"}, ["arg1"]))

Exercice 1 : a completer -> None


## Exercice 2 — Traquer une fuite dans un état assaini

**Objectif** : écrire `fuite_nominale(etat_sanitise, motifs)` qui renvoie la
liste des **motifs** (chaînes) qui apparaissent encore dans la sérialisation
JSON de l'état assaini. La réponse doit être une liste de motifs **trouvés**
(vide si l'état est propre).

**Attention** : la recherche doit porter sur la **sérialisation complète**,
pas sur les clés de premier niveau — c'est exactement le test que la cellule
[4] fait à la main.

**Indice** : `json.dumps(etat, ensure_ascii=False)` puis une recherche de
sous-chaîne par motif.

In [14]:
# Exercice 2 - a completer
def fuite_nominale(etat_sanitise, motifs):
    """Renvoie la liste des motifs qui subsistent dans l'etat assaini.

    TODO etudiant : faire le travail.
    """
    # TODO etudiant : implementer (serialisation JSON + recherche par motif).
    return []  # TODO etudiant


print("Exercice 2 : a completer ->", fuite_nominale({"texte": "clair"}, ["clair"]))

Exercice 2 : a completer -> []


## Exercice 3 — Comparer deux corpus par leur asymétrie

**Objectif** : écrire `comparer_corpus(sigs)` qui, pour chaque corpus d'une
liste de signatures, renvoie un couple `(nombre_de_sophismes, asymétrie)` — le
nombre de sophismes détectés et l'asymétrie tricherie/influence, calculée
comme `(influence - tricherie) / (influence + tricherie)`.

**Attention** : un corpus sans aucune tricherie doit produire une asymétrie de
`1.0` (convention du ratio, cf. le plafond `1e9` de la cellule [9]), et non
une division par zéro.

**Indice** : les tables `TRICHERIE_TYPES` et `INFLUENCE_TYPES` du module
`argumentation_lib._pattern_mining` classent les types ; le nombre de sophismes
se lit dans `state["identified_fallacies"]`.

In [15]:
# Exercice 3 - a completer
def comparer_corpus(sigs):
    """Renvoie, par corpus, le couple (nombre de sophismes, asymetrie).

    TODO etudiant : faire le travail.
    """
    # TODO etudiant : implementer (frequences par type + asymetrie).
    return {}  # TODO etudiant


print("Exercice 3 : a completer ->", comparer_corpus(SIGS))

Exercice 3 : a completer -> {}


## Récapitulatif et limites

| Organe | Contrat | Piège mesuré |
|---|---|---|
| `opaque_id` | déterministe, distinct, non réversible | un sel **public** en fait un oracle de confirmation ; sans sel, `RuntimeError` (#1973) |
| `sanitize_state` | les agrégats survivent, les textes partent | un conteneur **inconnu traverse intact** : l'allowlist est le contrat (#1664) |
| `pattern_mining` | des motifs sur signatures **assainies** | des attaques en dictionnaires font compter des entrées, pas des arêtes ; contrat tri-états `coherent` |

**Ce que ce carnet n'établit pas** :

- Il n'exerce que des **données synthétiques** : aucune conclusion sur un
  corpus réel, ni sur l'efficacité du garde en production.
- Le garde est une **allowlist**, pas une garantie : ce qu'il ne connaît pas
  traverse. La sécurité d'ensemble se gagne par l'**agrégation en amont**.
- `pattern_mining` n'est ici exercé que sur le chemin **sans formalisme Dung**
  de `run_formal_detectors` : le chemin `native_dung` de l'amont dépend d'un
  module qui n'est **pas** porté dans `argumentation_lib/` (cf. l'en-tête du
  fichier et `NOTICE-EPITA`).
- Le portage est **verbatim** : le comportement observé est donc celui de
  l'amont au commit `2e2a1c3c`, pas une réécriture.

**Suite** : le carnet 09 (garde-fous de pipeline LLM) porte le **contrat
d'échec** de l'entrée ; celui-ci porte le **contrat de sortie** de l'export.
Les deux se lisent ensemble : ce qui entre doit avoir été analysé, ce qui sort
doit avoir été dépouillé.